# Chen et al. (2024) — code validation

Goal: run **our** N-body / mass-loss pipeline (`kdk_leapfrog` -> `get_bound_particles`)
with the **paper's** setup and check we reproduce **Fig. 11** (arXiv:2408.01496).

Test case: $(M_0, R_\mathrm{apo}, e) = (10^5\,M_\odot,\ 40\ \mathrm{kpc},\ 0.6)$, in-plane,
MWPotential2014, King $W=8$ truncated at $r_\mathrm{tid}$, $\tau=2^{-13}$, $\epsilon=1$ pc,
$m=10\,M_\odot$, 3 Gyr.

Run top to bottom. **Step 1** is the quick (~5 min) accuracy check. **Step 2** (the full
21-run sweep, ~2.3 h cold, then cached) is optional. Launch Jupyter from this folder
(`ngc6569/`) so the relative paths work.

In [ ]:
import os, hashlib, json
from time import time
import numpy as np
import astropy.units as u
import agama

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize

from leap_frog import kdk_leapfrog
from bound_funcs import get_bound_particles
from cluster_utils import (shift_to_gc, trajectories, sample_king,
                           king_trunc_ratio, tidal_radius, apocenter_ic)

agama.setUnits(length=1*u.kpc, velocity=1*u.km/u.s, mass=1*u.Msun)
time_unit = u.kpc.to(u.km) * u.s.to(u.Gyr)   # 'kpc km^-1 s' -> Gyr
print("agama G =", agama.G)

In [ ]:
# --- paper parameters ---
FID = dict(W=8.0, kmax=13, eps_pc=1.0, m_star=10.0)
M0         = 1e5            # cluster mass [Msun]
R_APO      = 40.0          # apocenter [kpc]
ECC        = 0.6           # eccentricity
R_PERI     = R_APO * (1 - ECC) / (1 + ECC)   # = 10 kpc
T_FINAL    = -3000 * u.Myr  # 3 Gyr
DOWNSAMPLE = 20

pot_study = agama.Potential("milkyway/MWPotential2014.ini")

In [ ]:
center_pos, center_vel = apocenter_ic(pot_study, R_APO, R_PERI, time_unit)
_v_t = center_vel[1]

# verify eccentricity
_, _traj = agama.orbit(potential=pot_study, ic=[R_APO, 0, 0, 0, _v_t, 0],
                       time=3.0 / time_unit, trajsize=4000)
_rmin = np.sqrt((_traj[:, :3]**2).sum(axis=1)).min()
print(f"v_t={_v_t:.1f} km/s | min r={_rmin:.2f} kpc, e={(R_APO - _rmin)/(R_APO + _rmin):.3f}")

r_tid = tidal_radius(R_APO, M0, pot_study)
print(f"r_tid = {r_tid*1000:.1f} pc")
print(f"c(W=8) = r_t/r0 = {king_trunc_ratio(8.0):.3f}")

In [ ]:
Nbody = int(round(M0 / FID["m_star"]))
scaleRadius = r_tid / king_trunc_ratio(FID["W"])
xv, mass    = sample_king(FID["W"], scaleRadius, M0, Nbody)
pos_0, vel_0 = shift_to_gc(xv, center_pos, center_vel)

In [ ]:
tmax = -T_FINAL.to(u.Gyr).value / time_unit
print("maximum time", tmax)

In [ ]:
kmax = FID["kmax"]
tau = 2**(-kmax) * time_unit
print("time step:", tau)
nt = int(tmax / tau) + 1
print("number of time steps:", nt)

eps = FID["eps_pc"] / 1000.0
print("softening length:", eps)

downsample = DOWNSAMPLE

In [ ]:
t1 = time()
sim_data = kdk_leapfrog(pot_study, pos_0, vel_0, 
                        mass, nt, tau, agama.G, eps, 
                        time_unit, downsample, last_snapshot=False)
t2 = time()
print("run time", t2-t1, (t2-t1)/60)

In [ ]:
bound_data = get_bound_particles(sim_data, mass)

In [ ]:
traj = trajectories(bound_data, sim_data)

## Step 1 — fiducial accuracy check (~5 min)

One run at the paper's fiducial (W=8, $\tau=2^{-13}$, $\epsilon=1$ pc, $m=10$, N=10,000).
**Pass if M/M0 at 3 Gyr lands around 0.70-0.72** (the black curve in every Fig. 11 panel).

In [ ]:
_t_fid = traj["time"]                        # already in Gyr (converted in _create_snapshot)
_M_fid  = traj["mass"] / traj["mass"][0]
print(f"M/M0 at 3 Gyr = {_M_fid[-1]:.3f}   (paper fiducial ~0.70-0.72)")

plt.figure(figsize=(6, 4))
plt.plot(_t_fid, _M_fid, color="black", lw=2)
plt.axhspan(0.70, 0.72, color="green", alpha=0.15, label="paper endpoint")
plt.xlim(0, 3); plt.ylim(0.7, 1.0)
plt.xlabel("t (Gyr)"); plt.ylabel(r"$M(t)/M_0$")
plt.title("Fiducial check vs Chen Fig. 11")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
CACHE_DIR = "output/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def _cache_key(W, kmax, eps_pc, m_star, Nbody):
    payload = dict(W=float(W), kmax=int(kmax), eps_pc=float(eps_pc),
                   m_star=float(m_star), Nbody=int(Nbody), M0=float(M0),
                   R_APO=float(R_APO), ECC=float(ECC),
                   tfin_myr=float(T_FINAL.to(u.Myr).value),
                   downsample=int(DOWNSAMPLE), r_tid=round(float(r_tid), 9))
    h = hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:10]
    return f"{CACHE_DIR}/ml_W{W:g}_k{kmax}_eps{eps_pc:g}_m{m_star:g}_N{Nbody}_{h}.npz"

def run_mass_loss(W, kmax, eps_pc, m_star, use_cache=True):
    Nbody = int(round(M0 / m_star))
    key = _cache_key(W, kmax, eps_pc, m_star, Nbody)
    if use_cache and os.path.exists(key):
        d = np.load(key)
        return d["t"], d["M"]

    scaleRadius = r_tid / king_trunc_ratio(W)
    xv, mass    = sample_king(W, scaleRadius, M0, Nbody)
    pos, vel    = shift_to_gc(xv, center_pos, center_vel)

    tmax = -T_FINAL.to(u.Gyr).value / time_unit
    tau  = 2 ** (-kmax) * time_unit
    nt   = int(tmax / tau) + 1
    eps  = eps_pc / 1000.0

    sim_data   = kdk_leapfrog(pot_study, pos, vel, mass, nt, tau, agama.G, eps,
                              time_unit, DOWNSAMPLE, last_snapshot=False)
    bound_data = get_bound_particles(sim_data, mass)
    traj       = trajectories(bound_data, sim_data)
    t_gyr, M_over_M0 = traj["time"], traj["mass"] / traj["mass"][0]
    if use_cache:
        np.savez(key, t=t_gyr, M=M_over_M0)
    return t_gyr, M_over_M0

## Step 2 (optional) — full 4-panel parameter study

~2.3 h cold; every finished curve is cached, so re-runs replay in seconds.
$\epsilon=0$ (PeTar) is omitted; the $\tau$ panel is expected *not* to fully converge.

In [ ]:
results = {"W": {}, "tau": {}, "eps": {}, "m": {}}
for W in [2, 4, 6, 8, 10, 12, 14]:
    if np.isclose(W, FID["W"]):
        results["W"][W] = (_t_fid, _M_fid); continue
    print("W", W); results["W"][W] = run_mass_loss(W, FID["kmax"], FID["eps_pc"], FID["m_star"])
for k in [10, 11, 12, 13, 14, 15]:
    if k == FID["kmax"]:
        results["tau"][k] = (_t_fid, _M_fid); continue
    print("kmax", k); results["tau"][k] = run_mass_loss(FID["W"], k, FID["eps_pc"], FID["m_star"])
for e in [2**-2, 2**-1, 2**0, 2**1, 2**2]:
    if np.isclose(e, FID["eps_pc"]):
        results["eps"][e] = (_t_fid, _M_fid); continue
    print("eps", e); results["eps"][e] = run_mass_loss(FID["W"], FID["kmax"], e, FID["m_star"])
for ms in [2**-3*10, 2**0*10, 2**3*10]:
    if np.isclose(ms, FID["m_star"]):
        results["m"][ms] = (_t_fid, _M_fid); continue
    print("m", ms); results["m"][ms] = run_mass_loss(FID["W"], FID["kmax"], FID["eps_pc"], ms)
print("done")

In [ ]:
def _colors(values, fiducial):
    vals = sorted(values)
    norm = Normalize(vmin=min(vals), vmax=max(vals))
    return {v: ("black" if np.isclose(v, fiducial) else cm.coolwarm(norm(v))) for v in vals}

fig, axes = plt.subplots(4, 1, figsize=(6, 14), sharex=True)
fig.suptitle("Chen et al. 2024 (Fig. 11)", style="italic", size=16)

ax = axes[0]; cols = _colors(results["W"], FID["W"])
for W in sorted(results["W"]):
    t, M = results["W"][W]; ax.plot(t, M, color=cols[W], lw=1.5, label=fr"$W = {int(W)}$")

ax = axes[1]; cols = _colors(results["tau"], FID["kmax"])
for k in sorted(results["tau"]):
    t, M = results["tau"][k]
    ax.plot(t, M, color=cols[k], lw=1.5, label=fr"$\tau = 2^{{-{k}}}$ kpc km$^{{-1}}$ s")

ax = axes[2]; cols = _colors(results["eps"], FID["eps_pc"])
for e in sorted(results["eps"]):
    t, M = results["eps"][e]
    ax.plot(t, M, color=cols[e], lw=1.5, label=fr"$\epsilon = 2^{{{int(round(np.log2(e)))}}}$ pc")

ax = axes[3]; cols = _colors(results["m"], FID["m_star"])
for ms in sorted(results["m"]):
    t, M = results["m"][ms]
    ax.plot(t, M, color=cols[ms], lw=1.5,
            label=fr"$m = 2^{{{int(round(np.log2(ms/10)))}}} \times 10\ M_\odot$")

for ax in axes:
    ax.set_xlim(0, 3); ax.set_ylim(0.7, 1.0)
    ax.set_ylabel(r"$M(t)/M_0$", size=14)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=10)
axes[-1].set_xlabel(r"$t$ (Gyr)", size=14)
plt.tight_layout()
fig.savefig("output/chen_fig11_massloss.pdf", bbox_inches="tight")
plt.show()